# 第 3 课 · 激活、梯度、初始化、BatchNorm

## 你问过的问题（按出现顺序）

| # | 问题 | 答案要点 |
|---|---|---|
| 1 | b1 能不能设置成 0？ | 可以但 Karpathy 用 *0.01。**W 必须非零**（破对称），**b 可以 0**。**b2 应该 = 0**（避免对答案的无理偏见）。|
| 2 | 输入层能没有偏见、输出却要公正？ | 输入层 b1 后面经过 W2 重新混合，偏差被稀释。输出层 b2 后面没有混合，**直接进 softmax**，所以要公正。**越接近输出越敏感**。|
| 3 | W1 为啥要这样变（Kaiming 直觉）？ | pre-activation = 30 项独立求和，**标准差 = √30 × σ(W)**。要 pre-act std≈1，σ(W) 必须 = 1/√30。tanh 再补 gain=5/3。|
| 4 | 我代码 `** 0.5` 报 NaN？ | `**0.5` 写错位置作用到 randn 输出上→对负数开根号 → NaN。**正确**：作用在标量 `(n_embd*block_size)` 上。|
| 5 | `mean(0, keepdim=True)` 这两参数啥意思？ | `0` = 沿 batch 维求均值（每个神经元在 32 样本上的均值）；`keepdim=True` = 保留压缩维度为 1，便于广播 (32,200)→(1,200)。|
| 6 | evaluate 里那行 `bngain * (h - mean) / std + bias` 干啥？ | BatchNorm 反归一化公式。评估时用 `bn_running_mean/std`（训练全程的 EMA），训练时用 batch 自身的 mean/std。|
| 7 | bngain/bnbias/bn_running_* 这四个是啥？ | 前两个是**可学习**的反归一化 gain & bias（初始 1, 0）；后两个是**非学习**的 running stats（EMA 维护，评估时代替 batch stats，避免 batch_size=1 除 0）。|
| 8 | BatchNorm 不是这课的吧？ | **是**，Lesson 3 重点。但最终 Transformer 用 LayerNorm（沿 feature 维归一化，不依赖 batch）。这节课为 Lesson 6 LayerNorm 埋伏笔。|

## 重点（按修复顺序）

### ① 算"理论合理初始 loss"
- 随机模型最好输出均匀分布 → loss = -log(1/vocab_size)
- vocab=27 时 = **-log(1/27) ≈ 3.30**
- 偏离这个数字就是初始化有问题

### ② 修复 1：W2 *= 0.01, b2 = 0
- 让初始 logits 接近 0 → softmax ≈ 均匀 → loss ≈ 3.30
- **初始 loss 从 26 跳到 3.3**

### ③ 诊断 tanh 饱和
- `(h.abs() > 0.99).float().mean()` > 10% 就警惕
- tanh 饱和处梯度 = 1 - tanh² ≈ 0 → 神经元死掉

### ④ 修复 2：Kaiming 初始化 W1
- 公式：**σ(W) = gain / √fan_in**
- tanh gain = 5/3，fan_in = 30 → σ ≈ 0.304
- 训练后 dev loss：**2.34 → 2.14**

### ⑤ BatchNorm 实验
- 公式：`bngain * (h - μ) / σ + bnbias`
- 训练用 batch stats，评估用 running stats（EMA 维护）
- 插入位置：**Linear 之后、激活之前**
- **小心**：BN 在浅 MLP 上**没红利**——给后面 LayerNorm 埋伏笔

## 关键 PyTorch 招式

- `tensor.std()` / `tensor.min()` / `tensor.max()` —— 形状诊断三件套
- `(tensor.abs() > 阈值).float().mean()` —— 饱和率统计
- `tensor.mean(dim, keepdim=True)` —— 沿轴聚合，keepdim 保维度便于广播
- `with torch.no_grad():` —— 包住"不该进计算图的"统计更新
- `@torch.no_grad()` 装饰器 —— 整个评估函数关计算图

## 最终成绩

| 阶段 | train loss | dev loss |
|---|---|---|
| Lesson 2 末（病态初始化） | 2.32 | 2.34 |
| 修复 1（W2/b2） | 待测 | 待测 |
| 修复 2（+ Kaiming） | **2.09** | **2.14** ← 命中 Karpathy 视频水平 |
| + BatchNorm | 2.14 | 2.16 ← 略退步，预期内（浅 MLP）|

---

下面是详细笔记和可运行代码。

# 第 3 课 · 激活、梯度、初始化、BatchNorm

## 课程目标
诊断并修复 Lesson 2 留下的两个病灶，把 dev loss 从 2.34 推到 2.14。

## 病灶速查
| 病灶 | 表现 | 根因 | 修法 |
|---|---|---|---|
| 初始 loss = 26（应 ≈ 3.3） | 训练前期浪费 | W2/b2 太大 → logits 飘到 ±50 → softmax 极端 | W2 *= 0.01, b2 = 0 |
| tanh 饱和率 70%+ | 大半神经元梯度恒 0 | W1 太大 → pre-activation std ≈ √30 ≈ 5.5 | Kaiming: W1 *= (5/3)/√30 |

## 概念分层
1. **理论合理 loss = -log(1/27) ≈ 3.3**（均匀分布时的 NLL）
2. **方差传播规则**：N 个独立项相加，标准差 = √N × 单项标准差
3. **Kaiming 公式**：σ(W) = gain / √fan_in。tanh 的 gain = 5/3
4. **饱和判据**：`(h.abs() > 0.99).float().mean()` 超过 10% 就该警惕
5. **BatchNorm**：归一化 + 可学的 gain/bias，但在浅 MLP 上**红利近 0**——为 Lesson 6 LayerNorm 埋伏笔

## 最终成绩
- **初始化修复后**：train 2.09 / dev 2.14（≈ Karpathy 视频水平 2.10）
- **加 BatchNorm 后**：train 2.14 / dev 2.16（略退步，浅 MLP 上 BN 的真实表现）

## 1. 共享 setup（数据 + 数据集构造，一次性运行）

In [ ]:
import torch
import torch.nn.functional as F
import random

words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(stoi)        # 27

block_size = 3
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            X.append(context)
            Y.append(stoi[ch])
            context = context[1:] + [stoi[ch]]
    return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr,  Ytr  = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte,  Yte  = build_dataset(words[n2:])

print(f"Xtr {Xtr.shape}, Xdev {Xdev.shape}, Xte {Xte.shape}")

n_embd, n_hidden = 10, 200    # 嵌入维度、隐藏层宽度

## 2. 病灶 1：初始 loss 飙到 26

**理论合理值**：随机初始化的模型应该输出**均匀分布**（每个字符 1/27 概率），loss = -log(1/27) ≈ **3.30**。

**实际观察到 26**——8 倍偏离。原因：W2、b2 ~ N(0, 1) 让 logits 飘到 ±50，softmax 给某个错误字符 99.99% 信心 → -log(0.0001) 是巨大的数字。

In [ ]:
# 病态版初始化（Lesson 2 同款）
g = torch.Generator().manual_seed(2147483647)
C  = torch.randn((vocab_size, n_embd),            generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g)
b1 = torch.randn(n_hidden,                        generator=g)
W2 = torch.randn((n_hidden, vocab_size),          generator=g)
b2 = torch.randn(vocab_size,                      generator=g)

# 一次性 forward，诊断病情
emb = C[Xtr]
h = torch.tanh(emb.view(-1, n_embd * block_size) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytr)

print(f"logits 标准差: {logits.std().item():.2f}    (应 ≈ 1)")
print(f"logits 范围:   {logits.min().item():.2f} ~ {logits.max().item():.2f}    (应 ≈ ±3)")
print(f"初始 loss:     {loss.item():.4f}    (应 ≈ 3.30)")

## 3. 修复 1：缩小 W2、b2 = 0 → 初始 loss → 3.3

**思路**：让初始 logits 接近 0 → softmax 输出 ≈ 1/27 均匀 → loss ≈ 3.30。

为啥 W2 不能 = 0？全 0 → 所有神经元对称，破不掉对称性。**小但非零**是关键。

为啥 b2 应该 = 0？避免给某些字符无理偏见。（b1 设 0 也行，因为下游 W2 会重新混合）

In [ ]:
# 应用修复 1
g = torch.Generator().manual_seed(2147483647)
C  = torch.randn((vocab_size, n_embd),            generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g)
b1 = torch.randn(n_hidden,                        generator=g)
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.01    # ← 缩小 100 倍
b2 = torch.zeros(vocab_size)                                              # ← 直接 0

emb = C[Xtr]
h = torch.tanh(emb.view(-1, n_embd * block_size) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytr)

print(f"logits 标准差: {logits.std().item():.2f}")
print(f"初始 loss:     {loss.item():.4f}    (应 ≈ 3.30)")

## 4. 病灶 2：tanh 饱和率 62%

**问题**：W1 ~ N(0, 1) + emb 也 ~ N(0, 1) → 30 个独立项相加，pre-activation 标准差 ≈ √30 × 1 ≈ **5.5** → 喂进 tanh 被压成 ±1。

**后果**：tanh 在 ±1 处梯度 = 1 - tanh² = 0 → 这些神经元**死掉**，所有上游参数收不到更新信号。

**判据**：`(h.abs() > 0.99)` 比例超过 10% 就该警惕。

In [ ]:
# 诊断（仍用修复 1 后的参数）
emb = C[Xtr]
h = torch.tanh(emb.view(-1, n_embd * block_size) @ W1 + b1)

print(f"h 标准差:        {h.std().item():.4f}")
print(f"h 饱和率 (|h|>0.99):  {(h.abs() > 0.99).float().mean().item():.2%}    (应 ≈ 5-10%)")

## 5. 修复 2：Kaiming 初始化 W1

**公式**：

$$
\sigma(W) = \frac{\text{gain}}{\sqrt{\text{fan\_in}}}
$$

- `gain` 是激活函数的补偿系数：linear=1, sigmoid=1, **tanh=5/3**, ReLU=√2
- `fan_in` 是这层的输入维度（这里 30 = block_size × n_embd）

代入：σ(W1) = (5/3) / √30 ≈ **0.304** → pre-activation std 回到 ≈ 1 → tanh 工作在 sweet spot。

**易踩坑**：`** 0.5` 的位置——要作用在 `(n_embd * block_size)` 这个标量上，不是 `randn(...)` 上。否则对负随机数开根号会出 NaN。

In [ ]:
# 应用完整修复（W1 Kaiming + W2/b2）
g = torch.Generator().manual_seed(2147483647)
C  = torch.randn((vocab_size, n_embd),            generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3) / (n_embd * block_size)**0.5
b1 = torch.randn(n_hidden,                        generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.01
b2 = torch.zeros(vocab_size)

parameters = [C, W1, b1, W2, b2]
for p in parameters:
    p.requires_grad = True

print(f"参数总数: {sum(p.nelement() for p in parameters)}")

# 三诊断指标，全部应健康
emb = C[Xtr]
h = torch.tanh(emb.view(-1, n_embd * block_size) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytr)

print(f"初始 loss:         {loss.item():.4f}    (应 ≈ 3.30)")
print(f"h 标准差:          {h.std().item():.4f}    (应 ≈ 0.65)")
print(f"h 饱和率 (|h|>.99): {(h.abs() > 0.99).float().mean().item():.2%}    (应 ≈ 5-10%)")

## 6. 训练循环（用修好的初始化）

期望：训练全程 loss 平稳，**前期不再浪费在拉回 logits 量级上**。

In [ ]:
batch_size = 32

for step in range(20000):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,))
    Xb, Yb = Xtr[ix], Ytr[ix]

    # forward
    emb = C[Xb]
    h = torch.tanh(emb.view(-1, n_embd * block_size) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    # backward
    for p in parameters:
        p.grad = None
    loss.backward()

    # update（学习率分阶段）
    lr = 0.1 if step < 10000 else 0.01
    for p in parameters:
        p.data -= lr * p.grad

    if step % 2000 == 0:
        print(f"step {step:5d} | loss {loss.item():.4f}")

print(f"final batch loss = {loss.item():.4f}")

## 7. 评估真实 train / dev loss

mini-batch loss 噪声大不可信，必须在完整数据集上重算。

In [ ]:
@torch.no_grad()
def evaluate(split):
    X, Y = {'train': (Xtr, Ytr), 'dev': (Xdev, Ydev), 'test': (Xte, Yte)}[split]
    emb = C[X]
    h = torch.tanh(emb.view(-1, n_embd * block_size) @ W1 + b1)
    logits = h @ W2 + b2
    return F.cross_entropy(logits, Y).item()

print(f"train: {evaluate('train'):.4f}    (期望 ≈ 2.09)")
print(f"dev:   {evaluate('dev'):.4f}    (期望 ≈ 2.14)")

# 健康指标：dev - train ≈ 0.05 是健康的小幅过拟合
# > 0.3 严重过拟合，需要正则化 / 减小模型 / 加更多数据

## 8. 加入 BatchNorm（教学性实验）

### 思想
Kaiming 只保证"起点"健康。BN 进一步**强制每一步、每一层**都健康：

$$
\tilde{h} = \text{bngain} \cdot \frac{h - \mu_{\text{batch}}}{\sigma_{\text{batch}}} + \text{bnbias}
$$

### 四个新参数

| 名字 | 形状 | 学习 | 作用 |
|---|---|---|---|
| `bngain` | (1, 200) | ✅ | 反归一化缩放（初始 1） |
| `bnbias` | (1, 200) | ✅ | 反归一化平移（初始 0） |
| `bn_running_mean` | (1, 200) | ❌（训练时 EMA 更新） | 评估时代替 batch mean |
| `bn_running_std` | (1, 200) | ❌（同上） | 评估时代替 batch std |

### 训练 vs 评估
- **训练**：用 `bn_mean = h.mean(0, keepdim=True)`，当前 batch 的均值
- **评估**：用 `bn_running_mean`，训练全程的 EMA。否则 batch_size=1 时除 0 爆炸

### `mean(0, keepdim=True)` 拆解
- `0`：沿维度 0（batch 维）求均值 → 每个神经元的"batch 平均"
- `keepdim=True`：保留压缩的维度为 1，shape (32, 200) → (1, 200)，方便广播

### 插入位置
**Linear 之后、激活之前**。因为 BN 会减掉 mean，所以 b1 变得冗余（加了也会被减掉）。

### 8.1 重新初始化（加入 BN 参数）

In [ ]:
g = torch.Generator().manual_seed(2147483647)
C  = torch.randn((vocab_size, n_embd),            generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3) / (n_embd * block_size)**0.5
b1 = torch.randn(n_hidden,                        generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.01
b2 = torch.zeros(vocab_size)

# BN 可学习参数
bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))

# BN 运行统计量（非可学习，但训练时更新）
bn_running_mean = torch.zeros((1, n_hidden))
bn_running_std  = torch.ones((1, n_hidden))

# 注意：running 统计量不进 parameters 列表
parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad = True

print(f"参数总数: {sum(p.nelement() for p in parameters)}")

### 8.2 训练循环（forward 里插 BN）

In [ ]:
batch_size = 32

for step in range(20000):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,))
    Xb, Yb = Xtr[ix], Ytr[ix]

    # forward
    emb = C[Xb]
    h_preact = emb.view(-1, n_embd * block_size) @ W1 + b1

    # ---- BatchNorm ----
    bn_mean = h_preact.mean(0, keepdim=True)
    bn_std  = h_preact.std(0, keepdim=True)
    h_preact = bngain * (h_preact - bn_mean) / bn_std + bnbias

    # EMA 更新 running stats（必须 no_grad 否则会进计算图）
    with torch.no_grad():
        bn_running_mean = 0.999 * bn_running_mean + 0.001 * bn_mean
        bn_running_std  = 0.999 * bn_running_std  + 0.001 * bn_std
    # ---- BN end ----

    h = torch.tanh(h_preact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    # backward
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    lr = 0.1 if step < 10000 else 0.01
    for p in parameters:
        p.data -= lr * p.grad

    if step % 2000 == 0:
        print(f"step {step:5d} | loss {loss.item():.4f}")

### 8.3 评估（用 running stats，不用 batch stats）

In [ ]:
@torch.no_grad()
def evaluate_bn(split):
    X, Y = {'train': (Xtr, Ytr), 'dev': (Xdev, Ydev), 'test': (Xte, Yte)}[split]
    emb = C[X]
    h_preact = emb.view(-1, n_embd * block_size) @ W1 + b1
    # 评估用 running stats，不能用 batch stats（评估可能只有 1 个样本，除 0）
    h_preact = bngain * (h_preact - bn_running_mean) / bn_running_std + bnbias
    h = torch.tanh(h_preact)
    logits = h @ W2 + b2
    return F.cross_entropy(logits, Y).item()

print(f"train: {evaluate_bn('train'):.4f}")
print(f"dev:   {evaluate_bn('dev'):.4f}")

## 9. 复盘：BatchNorm 为啥没帮上忙？

### 观察
- 修初始化后：train **2.09** / dev **2.14**
- 加 BN 后：train **2.14** / dev **2.16**——**略退步**

### 原因
BN 的红利**主要在深网络**：5 层、10 层、50 层那种，激活值会层层放大/缩小，BN 帮你逐层校准。

但我们的网络**只有 1 层 hidden**——激活只走一次，没有层层累积问题，**BN 的"校准"红利接近 0**。

而 BN 同时引入了**自带噪声**：每 batch 只有 32 个样本算 mean/std，自带约 17%（1/√32）相对随机误差。这扰动在浅模型上**没被红利抵消**，反而拖后腿。

### 关键认知
1. **BN 不是万能升级**。它是针对特定问题（深网络激活漂移）的工具。
2. **LayerNorm 改沿 feature 维归一化**，不依赖 batch → 没有"32 样本统计噪声"问题。
3. Transformer 用 LayerNorm 不用 BN 的原因，你现在亲身体会过了——到 Lesson 6 看会有共鸣。



## 10. 完整代码

In [ ]:
import torch
import torch.nn.functional as F
import random

# ============================================================
# 1. 数据准备
# ============================================================
words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(stoi)        # 27

block_size = 3
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            X.append(context)
            Y.append(stoi[ch])
            context = context[1:] + [stoi[ch]]
    return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr,  Ytr  = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte,  Yte  = build_dataset(words[n2:])

# ============================================================
# 2. 参数初始化（修好的版本：Kaiming W1 + 小 W2 + 0 b2 + BN）
# ============================================================
n_embd, n_hidden = 10, 200
g = torch.Generator().manual_seed(2147483647)

C  = torch.randn((vocab_size, n_embd),            generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3) / (n_embd * block_size)**0.5
b1 = torch.randn(n_hidden,                        generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.01
b2 = torch.zeros(vocab_size)

# BatchNorm 参数
bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
bn_running_mean = torch.zeros((1, n_hidden))
bn_running_std  = torch.ones((1, n_hidden))

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad = True
print(f"参数总数: {sum(p.nelement() for p in parameters)}")

# ============================================================
# 3. 初始诊断（验证修复有效）
# ============================================================
emb = C[Xtr]
h_preact = emb.view(-1, n_embd * block_size) @ W1 + b1
h = torch.tanh(h_preact)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytr)
print(f"初始 loss: {loss.item():.4f}    (期望 ≈ 3.30)")
print(f"h 标准差:  {h.std().item():.4f}    (期望 ≈ 0.65)")
print(f"h 饱和率: {(h.abs() > 0.99).float().mean().item():.2%}    (期望 ≈ 5-10%)")

# ============================================================
# 4. 训练循环（mini-batch + BatchNorm + lr 分段）
# ============================================================
batch_size = 32

for step in range(20000):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,))
    Xb, Yb = Xtr[ix], Ytr[ix]

    # forward
    emb = C[Xb]
    h_preact = emb.view(-1, n_embd * block_size) @ W1 + b1

    # BatchNorm
    bn_mean = h_preact.mean(0, keepdim=True)
    bn_std  = h_preact.std(0, keepdim=True)
    h_preact = bngain * (h_preact - bn_mean) / bn_std + bnbias
    with torch.no_grad():
        bn_running_mean = 0.999 * bn_running_mean + 0.001 * bn_mean
        bn_running_std  = 0.999 * bn_running_std  + 0.001 * bn_std

    h = torch.tanh(h_preact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    # backward
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    lr = 0.1 if step < 10000 else 0.01
    for p in parameters:
        p.data -= lr * p.grad

    if step % 2000 == 0:
        print(f"step {step:5d} | loss {loss.item():.4f}")

print(f"final batch loss = {loss.item():.4f}")

# ============================================================
# 5. 评估（用 running stats，不用 batch stats）
# ============================================================
@torch.no_grad()
def evaluate(split):
    X, Y = {'train': (Xtr, Ytr), 'dev': (Xdev, Ydev), 'test': (Xte, Yte)}[split]
    emb = C[X]
    h_preact = emb.view(-1, n_embd * block_size) @ W1 + b1
    h_preact = bngain * (h_preact - bn_running_mean) / bn_running_std + bnbias
    h = torch.tanh(h_preact)
    logits = h @ W2 + b2
    return F.cross_entropy(logits, Y).item()

print(f"\ntrain: {evaluate('train'):.4f}")
print(f"dev:   {evaluate('dev'):.4f}")
